# Denisha Gemma 4 Developer Agent — local run

Run all cells in order. This notebook performs local validation and builds the two Kaggle submission ZIPs. It does not download or fine-tune the 31B model on the PC; Kaggle performs the official model evaluation.


In [ ]:
from pathlib import Path
import os, subprocess, sys

def find_project():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for base in candidates:
        p = base / 'Denisha_Gemma4_Agents'
        if (p / 'scripts' / 'smoke_test.py').exists():
            return p
    clone_root = Path.home() / 'GenAI'
    if not (clone_root / 'Denisha_Gemma4_Agents' / 'scripts' / 'smoke_test.py').exists():
        subprocess.run(['git', 'clone', 'https://github.com/denishatank12/GenAI.git', str(clone_root)], check=True)
    return clone_root / 'Denisha_Gemma4_Agents'

PROJECT = find_project()
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Python:', sys.version.split()[0])
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)


In [ ]:
# Install the small local validation dependency.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')


In [ ]:
# GPU visibility check. A 4090 is useful for local development, but official scoring runs on Kaggle.
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True, capture_output=True)
    print(result.stdout.strip() or result.stderr.strip())
except FileNotFoundError:
    print('nvidia-smi was not found. Continue only if this PC is not expected to provide CUDA.')


In [ ]:
# Fail early if either agent configuration is invalid.
subprocess.run([sys.executable, 'scripts/smoke_test.py'], check=True)
print('Smoke tests passed.')


In [ ]:
# Build both independent submission archives.
for variant in ('agent_1_tool_policy', 'agent_2_integrated'):
    subprocess.run([sys.executable, 'scripts/build_submission.py', '--variant', variant], check=True)
print('Build complete.')


In [ ]:
# Final artifact check: Kaggle requires agent.yaml at the ZIP root.
import zipfile
for variant in ('agent_1_tool_policy', 'agent_2_integrated'):
    archive = Path('build') / f'{variant}.submission.zip'
    with zipfile.ZipFile(archive) as zf:
        names = zf.namelist()
        assert 'agent.yaml' in names, f'agent.yaml missing from {archive}'
    print('READY:', archive.resolve(), f'({archive.stat().st_size:,} bytes)')
print('Both submission files are ready for Kaggle upload.')
